# HW4: Bayesian Inference, Gaussian Processes, and Bayesian Optimization (Spring 2026)

**Due:** April 9, 2026 at 23:59  |  **Total Points:** 100 (+ 6 bonus)

| Part | Topic | Points |
|---|---|---|
| 1 | Bayesian Inference + Information Theory | 30 |
| 2 | Gaussian Process Fundamentals | 20 |
| 3 | Bayesian Optimization by Hand | 35 |
| 4 | Bayesian Hyperparameter Tuning (Optuna) | 15 |

**Read `HW4_instructions.md`** for the full story, task requirements, and rubric.

## Environment Setup

```bash
cd HW/HW_spring_2026/HW4
uv sync
source .venv/bin/activate   # Windows: .venv\\Scripts\\activate
jupyter notebook HW4_spring2026.ipynb
```

## Submission Rules

- Use `random_state=88` everywhere randomness is involved.
- Do **not** use `plt.show()` -- save all figures with `plt.savefig()`.
- All **written answers** go in the designated markdown answer cells below.
  Do not put discussion in code comments.
- Submit this notebook OR `hw4_yourname.py` + `hw4_yourname_answers.md`,
  plus all required `.png` files.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # Saves figures to .png files without opening a window.
                        # ⚠️  REMOVE THIS LINE if running interactively in Jupyter --
                        #     otherwise plt.show() and inline plots will not render.
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.special import betaln, digamma
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    RBF, Matern, ExpSineSquared, RationalQuadratic,
    WhiteKernel, ConstantKernel,
)
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import optuna

RANDOM_STATE = 88
rng = np.random.default_rng(RANDOM_STATE)


---
# Part 1: Bayesian Inference and Information Theory Foundations (30 pts)

**Story:** You have joined a group screening YSZ thermal barrier coatings.
Each coating either survives 100 thermal cycles at 1100 degC (pass) or spalls (fail).
Tasks 1.1-1.9 build the full Bayesian and information-theoretic toolkit
that underlies GP regression and Bayesian Optimization in Parts 2-3.

See **HW4_instructions.md -- Part 1** for the complete scenario and requirements.

## Task 1.1: Bayesian Updating with a Conjugate Prior (4 pts)

Track your belief about the YSZ pass rate theta as 10 test results arrive.
Prior: Beta(3, 3). Data: [1, 1, 0, 1, 0, 0, 1, 1, 1, 0].

**Deliverables:** `bayes_posterior_update.png`, printed posterior statistics,
written answers to 3 questions.

In [ ]:
# Task 1.1 -- Bayesian Updating
# Your code here


### Task 1.1 -- Written Answers

**Q1: Why is the Beta distribution a natural choice for modeling a pass rate theta?
Address the support of the distribution and what conjugacy with the Bernoulli likelihood means.**

*Your answer here*

---

**Q2: The posterior mean is pulled toward the prior mean relative to the MLE.
Is this a bug or a feature in a small-data setting? What happens as n -> infinity?**

*Your answer here*

---

**Q3: State the correct interpretation of a 95% credible interval
and a 95% frequentist confidence interval.**

*Your answer here*

## Task 1.2: Prior Sensitivity Analysis (3 pts)

Three researchers, same 10 results, different priors:
Beta(1,1) (new student), Beta(3,3) (you), Beta(10,2) (postdoc with biased prior).

**Deliverables:** `bayes_prior_sensitivity.png`, written answers to 3 questions.

In [ ]:
# Task 1.2 -- Prior Sensitivity
# Your code here


### Task 1.2 -- Written Answers

**Q1: How different are the three posteriors after 10 observations?
Characterize quantitatively (posterior means and 95% CIs).**

*Your answer here*

---

**Q2: The postdoc's Beta(10,2) has effective strength equal to 10 virtual observations.
How many additional real experiments would they need before converging
close to the data-driven estimate?**

*Your answer here*

---

**Q3: Describe a materials domain scenario where you would
(a) confidently use an informative prior, and
(b) deliberately use a weakly informative or uninformative prior.**

*Your answer here*

## Task 1.3: From Parameters to Functions -- The Predictive Distribution (3 pts)

Fit a Bayesian linear model for thermal conductivity kappa vs. porosity phi:
y = w0 + w1*x + eps,  w ~ N(0, I),  sigma = 0.3.
Implement the closed-form posterior update yourself -- do not use a library.

**Deliverables:** `bayes_prior_predictive.png`, `bayes_posterior_predictive.png`,
written answers to 3 questions.

---

**Derivation hint — Bayesian linear regression posterior:**

Build a design matrix **Φ** of shape *(n × 2)* whose rows are `[1, xᵢ]` for each
observation. With prior **w** ~ N(**0**, **I**) and likelihood
**y** | **Φ**, **w** ~ N(**Φw**, σ²**I**), the closed-form posterior is:

```
Posterior covariance:  Σ_w = (ΦᵀΦ / σ² + I)⁻¹          # np.linalg.inv(...)
Posterior mean:        μ_w = Σ_w @ Φᵀ @ y / σ²

Predictive mean at x*:      μ*(x*) = φ(x*)ᵀ μ_w
Predictive variance at x*:  σ²*(x*) = φ(x*)ᵀ Σ_w φ(x*) + σ²
```

where `φ(x*) = np.array([1, x*])`. To draw sample lines from the posterior:
sample **w** ~ N(**μ_w**, **Σ_w**) via `rng.multivariate_normal(mu_w, Sigma_w)`
and plot `w[0] + w[1] * x_grid` for each sample.


In [ ]:
# Task 1.3 -- Bayesian Linear Regression (kappa-phi model)
# Your code here


### Task 1.3 -- Written Answers

**Q1: Describe what changed between the prior and posterior samples.
What specifically constrained the lines?**

*Your answer here*

---

**Q2: If your 5 measurements were all in the middle porosity range, how would
the model behave when extrapolating to very low porosity?
What would predictive uncertainty look like there?**

*Your answer here*

---

**Q3: A GP replaces the 2D weight vector with an infinite-dimensional function.
What do you expect the GP posterior to look like far from vs. near training data?**

*Your answer here*

## Task 1.4: Entropy, Conditional Entropy, and Joint Entropy (4 pts)

Compute differential entropy of the Beta posterior at each of the 10 stages.
Discretize into 200 bins and compare entropy for all 3 priors from Task 1.2.
Numerically verify the chain rule H(theta, data) = H(data) + H(theta | data).

**Deliverables:** `entropy_vs_observations.png`, printed entropy table,
written answers to 3 questions.

In [ ]:
# Task 1.4 -- Entropy, Conditional Entropy, Joint Entropy
# Your code here


### Task 1.4 -- Written Answers

**Q1: After five experiments, entropy is still relatively high.
What are two distinct reasons -- one related to your prior and one to the data observed?**

*Your answer here*

---

**Q2: Entropy is a property of one distribution.
KL divergence (Task 1.5) is asymmetric.
When comparing posterior to prior, why does direction matter
in a way entropy alone cannot capture?**

*Your answer here*

---

**Q3: Differential entropy can be negative.
Construct a simple example and explain physically what negative uncertainty means.**

*Your answer here*

## Task 1.5: KL Divergence and f-Divergences (5 pts)

Track KL(posterior_t || prior) across the 10-stage YSZ campaign.
Compute both KL directions, chi2/Hellinger/TV distances,
and JSD between the 3 priors.

**Deliverables:** `kl_divergence_curve.png`, printed divergence values,
written answers to 3 questions.

In [ ]:
# Task 1.5 -- KL Divergence and f-Divergences
# Your code here


### Task 1.5 -- Written Answers

**Q1: In variational inference you minimize KL(q || p) -- the reverse KL.
What distributional behavior does this encourage vs. minimizing the forward KL?
Why does this matter when the true posterior has multiple modes?**

*Your answer here*

---

**Q2: Hellinger distance is bounded in [0, 1]; KL is unbounded.
Give a concrete materials-science scenario where KL's unboundedness causes problems,
and explain why a bounded divergence would be preferable.**

*Your answer here*

---

**Q3: After 10 coating tests you compute KL(posterior || prior) = 0.02.
What would you tell your PI about whether those experiments were informative?**

*Your answer here*

## Task 1.6: Cross-Entropy, NLL, and Proper Scoring Rules (4 pts)

Score each researcher's sequential predictions using NLL (log score) and Brier score.
Use the predictive distribution *before* each observation arrives (no data leakage).

**Deliverables:** `scoring_rules.png`, printed cumulative NLL and Brier scores,
written answers to 4 questions.

In [ ]:
# Task 1.6 -- Cross-Entropy, NLL, Proper Scoring Rules
# Your code here


### Task 1.6 -- Written Answers

**Q1: The log score penalizes confident wrong predictions far more than uncertain ones.
Give one scenario where this is desirable and one where it could be dangerous.**

*Your answer here*

---

**Q2: Brier score is bounded in [0,1]; NLL is not.
What practical consequence does this have when comparing models
across datasets of different sizes or base rates?**

*Your answer here*

---

**Q3: Cross-entropy decomposes as H(P, Q) = H(P) + KL(P || Q).
During neural network training, P is fixed.
What does minimizing cross-entropy actually minimize,
and what part is irreducible regardless of model quality?**

*Your answer here*

---

**Q4: Researcher 3 predicted ~80% pass probability but was only right 60% of the time.
What does this tell you? Describe how to construct a calibration diagram,
and whether it is possible to be well-calibrated but have a high NLL.**

*Your answer here*

## Task 1.7: Mutual Information, Conditional MI, and Expected Information Gain (4 pts)

After 9 tests and one remaining, decide which future measurement reduces uncertainty most.
Track MI(theta; next_obs | data) across 10 stages.
Compute EIG for 50 candidate porosity values from the kappa-phi model.

**Deliverables:** `mutual_information_curve.png`, `eig_vs_sigma.png`,
written answers to 4 questions.

In [ ]:
# Task 1.7 -- Mutual Information and Expected Information Gain
# Your code here


### Task 1.7 -- Written Answers

**Q1: MI is symmetric: MI(X;Y) = MI(Y;X).
Does this mean observing Y is equally informative about X as observing X is about Y?
What does symmetry mean here vs. the physical act of measuring?**

*Your answer here*

---

**Q2: Looking at your eig_vs_sigma.png: what is the relationship between EIG and sigma(x)?
Is EIG just another name for sigma(x), or is there a meaningful difference?**

*Your answer here*

---

**Q3: Two candidate experiments: one in a high-sigma uncharted region,
one near a theoretically interesting but already-sampled composition (low sigma).
How does EIG inform the choice?
What does EIG fail to account for that a domain expert might override it on?**

*Your answer here*

---

**Q4: Give a concrete materials example where
MI(property; feature_A | feature_B) = 0 --
feature A carries no additional information once you know feature B.**

*Your answer here*

## Task 1.8: Differential Entropy of Gaussians and the ELBO (3 pts)

Compute posterior entropy h(x) at 50 porosity values from the kappa-phi model.
Compare joint entropy of 5 selected points vs. sum of their marginal entropies.
Identify ELBO terms for a Gaussian approximation to the Beta posterior.

**Deliverables:** `gp_posterior_entropy.png`, printed joint vs. marginal entropy,
written answers.

In [ ]:
# Task 1.8 -- Gaussian Differential Entropy and ELBO
# Your code here


### Task 1.8 -- Written Answers

**ELBO term identification:** Write out the two ELBO terms for a Gaussian q(theta)
approximating the Beta posterior from Task 1.1.
What does each term push q to do? Why is the Gaussian a poor choice here?

*Your answer here*

---

**Q1: If q is arbitrarily flexible (e.g. a normalizing flow), what happens to the KL term
as q -> posterior?
What is the practical consequence of removing the KL term entirely?**

*Your answer here*

---

**Q2: The GP log marginal likelihood (LML, Task 2.4) is the exact log evidence.
How does the ELBO relate to the LML?
When would you use the ELBO instead of the LML for a GP model?**

*Your answer here*

---

**Q3: Entropy search selects the next experiment by maximizing
EIG(x) = 0.5 * log[ det(Sigma_with_x) / det(Sigma_without_x) ].
Derive this from the EIG definition and the Gaussian differential entropy formula.
What assumption about the noise model makes this tractable?**

*Your answer here*

## [Bonus] Task 1.9: Fisher Information and the Cramer-Rao Bound (+3 pts)

Fisher information I(theta) measures expected log-likelihood curvature.
The Cramer-Rao bound Var(theta_hat) >= 1/I(theta) is a fundamental floor on estimator variance.

**Deliverables:** analytical derivation (markdown cell below),
`fisher_vs_entropy.png`, written answers to 3 questions.

> **Reminder:** The step-by-step derivation of I(theta) must go in the
> **markdown cell directly below** (Task 1.9 -- Fisher Information Derivation),
> not inside code comments or `print()` statements.
> The code cell produces `fisher_vs_entropy.png`; the derivation is separate.
>
> **matplotlib note:** `matplotlib.use('Agg')` is set in the import cell --
> correct for saving `.png` files, but remove it if running interactively in Jupyter.


### Task 1.9 -- Fisher Information Derivation

*Write your step-by-step derivation of I(theta) for the Bernoulli likelihood here.*

In [ ]:
# Task 1.9 (Bonus) -- Fisher Information and Cramer-Rao Bound
# Your code here


### Task 1.9 -- Written Answers

**Q1: Fisher information is additive: I_n(theta) = n * I_1(theta).
What does this imply about how posterior variance shrinks with sample size?
Is this consistent with your entropy curve from Task 1.4?**

*Your answer here*

---

**Q2: Natural gradient descent scales the update by I(theta)^-1.
Why is this preferable to standard gradient descent for models where
parameter space has non-Euclidean geometry?**

*Your answer here*

---

**Q3: Fisher information measures how quickly the likelihood changes with theta.
How does this connect to the LML curvature in Task 2.4?
Are they measuring the same thing?**

*Your answer here*

---
# Part 2: Gaussian Process Fundamentals (20 pts)

**Story:** You are mapping elastic modulus across the Al-Cu-Mg ternary alloy system.
Each composition costs ~$400 and 4 days. With 30-50 experiments in a semester,
you need a model that delivers both a predicted mean *and* calibrated uncertainty
at every candidate composition. That model is a Gaussian Process.

See **HW4_instructions.md -- Part 2** for the complete scenario and requirements.

## Task 2.1: GP as a Prior Over Functions (5 pts)

Before any measurements, choose a kernel that encodes your physical intuition.
Sample 5 function draws from each of 5 kernels and visualize the kernel matrices.

**Deliverables:** `gp_prior_samples.png` (5-panel), `gp_kernel_heatmaps.png`,
written answers to 4 questions.

In [ ]:
# Task 2.1 -- GP Prior Samples and Kernel Heatmaps
# Kernels: RBF, Matern(nu=0.5), Matern(nu=2.5), ExpSineSquared, RationalQuadratic
# Hint: call gp.sample_y(X_grid, n_samples=5) on an UNFITTED GPR instance
# Your code here


### Task 2.1 -- Written Answers

**Q1: What does the length scale control in the RBF kernel?
What happens to function samples as length_scale -> 0 or -> infinity?**

*Your answer here*

---

**Q2: The Matern family is parameterized by nu. What does nu control?
Why is nu=2.5 the standard choice for BO in materials science?**

*Your answer here*

---

**Q3: Which kernel would you choose for a property expected to vary periodically
with composition? With temperature?**

*Your answer here*

---

**Q4: What does it mean when two points in the kernel heatmap have high covariance?
Low covariance?**

*Your answer here*

## Task 2.2: Kernel Engineering -- Composite and Anisotropic Kernels (5 pts)

After 15 measurements you notice two-scale variation and anisotropic feature sensitivity.
Fit 4 kernels to a 1D synthetic dataset f(x) = sin(x) + 0.3*sin(5x), sigma_noise=0.15:
- K1: RBF(length_scale=1.0)
- K2: Matern(nu=2.5)
- K3: RBF() + WhiteKernel()
- K4: RBF(length_scale=2.0) + RBF(length_scale=0.3)

Also fit isotropic vs. ARD GP to the crossed-barrel dataset.

**Deliverables:** `gp_kernel_comparison.png`, printed hyperparameters+LML,
written answers to 3 questions.

In [ ]:
# Task 2.2 -- Kernel Engineering
# 1D synthetic: f(x) = sin(x) + 0.3*sin(5x), 15 noisy points, sigma_noise=0.15
# Use n_restarts_optimizer=5 for all fits
# Your code here


### Task 2.2 -- Written Answers

**Q1: When would you add kernels vs. multiply them?**

*Your answer here*

---

**Q2: What physical knowledge could guide kernel choice for bandgap,
elastic modulus, or thermal conductivity?**

*Your answer here*

---

**Q3: Why does ARD matter for high-dimensional input spaces
like CBFV feature vectors (100+ features)?**

*Your answer here*

## Task 2.3: The Three Sources of Uncertainty (5 pts)

The GP sigma(x) conflates aleatoric (irreducible) and epistemic (reducible) uncertainty.
Use f(x) = x*sin(x) over [0, 10] with Matern(nu=2.5) + WhiteKernel to separate them.
Also demonstrate hyperparameter sensitivity with 10 random initializations.

**Deliverables:** `gp_uncertainty_decomposition.png`,
`gp_hyperparameter_sensitivity.png`, written answers to 4 questions.

In [ ]:
# Task 2.3 -- Uncertainty Decomposition
# f(x) = x*sin(x), 10 noisy points over [0, 10], sigma_n=0.5
# Kernel: Matern(nu=2.5) + WhiteKernel(), normalize_y=True
# Your code here


### Task 2.3 -- Written Answers

**Q1: You observe high sigma(x) at a candidate point. Should you evaluate it?
Does your answer depend on whether the uncertainty is aleatoric or epistemic?**

*Your answer here*

---

**Q2: Your GP noise parameter converges to near zero.
Is this a good sign or a warning sign?**

*Your answer here*

---

**Q3: In a materials synthesis campaign, give concrete examples
of aleatoric and epistemic uncertainty.**

*Your answer here*

---

**Q4: Does sklearn's standard GP sigma(x) account for hyperparameter uncertainty?
What are the practical consequences in a BO campaign?**

*Your answer here*

## Task 2.4: Log Marginal Likelihood and Kernel Selection (5 pts)

The LML (log marginal likelihood) is how sklearn picks hyperparameters --
it balances fit quality against model complexity automatically.
Sweep the RBF length scale over 30 log-spaced values and plot the 1D LML curve.
Also compute a 2D LML surface over (length_scale, noise_level).

**Deliverables:** `gp_lml_curve.png`, `gp_lml_surface.png`,
written answers to 3 questions.

In [ ]:
# Task 2.4 -- Log Marginal Likelihood
# Use the 1D synthetic dataset from Task 2.2
# 1D sweep: RBF length_scale from 0.01 to 100 (30 log-spaced values)
# 2D surface: sweep length_scale and noise_level on 20x20 log grid
# Your code here


### Task 2.4 -- Written Answers

**Q1: What happens to the LML data-fit term and complexity term as length_scale -> 0?
Relate your answer to what you would observe when predicting modulus
at unmeasured compositions.**

*Your answer here*

---

**Q2: Five kernels all have training RMSE within 0.5 GPa.
How would you use LML to make a principled choice between them?**

*Your answer here*

---

**Q3: What is a practical failure mode of LML-based kernel selection when your
15 Al-Cu-Mg measurements are all clustered in one corner of the composition triangle?**

*Your answer here*

---
# Part 3: Bayesian Optimization by Hand (35 pts)

**Story:** StructureLab is developing 3D-printed crossed-barrel polymer lattice
crash absorbers for EV battery protection. Their FEA team ran ~1800 simulations
sweeping 4 geometric parameters (n, theta, r, t) to compute toughness.
Physical impact tests cost $150 and 3 days each. Budget: 150 tests.
Your goal: find at least 15 designs from the top 5% using BO.

**Rule:** Treat toughness as unknown. You can only evaluate a candidate
by looking up its value from the table -- simulating the cost of a physical test.

See **HW4_instructions.md -- Part 3** for the complete scenario and requirements.

## Task 3.1: Understand the Design Space (Tasks 3.1-3.3 graded together: 4 pts)

Load `data/crossed_barrel_dataset.csv`. Print summary statistics,
compute the 95th-percentile toughness threshold, and answer:
how many tests would random guessing require to find 15 top-5% candidates on average?

In [ ]:
# Task 3.1 -- Explore the Crossed-Barrel Dataset
df_cb = pd.read_csv('data/crossed_barrel_dataset.csv')
# Your code here


### Task 3.1 -- Written Answer

**Given random guessing, how many tests would you expect before finding
15 top-5% candidates? Show your reasoning.**

*Your answer here*

## Task 3.2: Initialize the Campaign

Randomly select 5 initial samples (`random_state=88`) as the seed observed set.
Everything else becomes the candidate pool.
Print the 5 initial designs and note how many are already in the top 5%.

In [ ]:
# Task 3.2 -- Initialize Campaign
FEATURE_COLS = ['n', 'theta', 'r', 't']
TARGET_COL = 'toughness'
# Your code here


## Task 3.3: Build the Gaussian Process Surrogate

Implement `fit_gp(X_train, y_train)` and `predict_gp(gp, scaler, X_candidates)`.
Use `Matern(nu=2.5) + ConstantKernel` with an internal `StandardScaler`.
Both functions must have inline comments on non-obvious steps.
Test: verify sigma > 0 for at least some candidates.

In [ ]:
# Task 3.3 -- GP Surrogate Functions

def fit_gp(X_train, y_train):
    # Normalize features so all 4 parameters contribute equally to the kernel
    # Return both fitted GP and scaler for use in predict_gp
    # Your code here
    pass


def predict_gp(gp, scaler, X_candidates):
    # Apply the saved scaler -- never refit on candidate data
    # Return mu and sigma as 1D numpy arrays
    # Your code here
    pass


# Test on initial 5 samples
# Your code here


## Task 3.4: Implement Upper Confidence Bound (UCB) (5 pts)

```
UCB(x) = mu(x) + kappa * sigma(x)
```

Implement `ucb` with inline comments.
Run a kappa sensitivity analysis for kappa in {0.01, 0.5, 2.0, 5.0, 20.0}:
print the toughness of the top-ranked candidate and the fraction of top-5% designs
in the top-10 UCB-ranked candidates.

In [ ]:
# Task 3.4 -- UCB Acquisition Function

def ucb(mu, sigma, kappa=2.0):
    # kappa=0 -> pure exploitation (highest predicted mean)
    # large kappa -> pure exploration (highest uncertainty)
    # Your code here
    pass


# Kappa sensitivity analysis
# Your code here


### Task 3.4 -- Written Answers

**Q1: At kappa=0.01, which designs does UCB select and why?
What is the risk of this strategy after only 5 observations?**

*Your answer here*

---

**Q2: At kappa=20.0, which designs does UCB select?
Is the top-ranked candidate likely to be in the top 5%?**

*Your answer here*

---

**Q3: Is there a principled way to set kappa, or is it always a manual choice?
(See Task 3.7 for the adaptive formula.)**

*Your answer here*

## Task 3.5: Implement Expected Improvement (EI) (3 pts)

```
Z     = (mu(x) - f_best - xi) / sigma(x)
EI(x) = (mu(x) - f_best - xi) * Phi(Z)  +  sigma(x) * phi(Z)
EI(x) = 0  if sigma(x) = 0
```

Implement `expected_improvement` with inline comments.
Guard the sigma==0 case to avoid division by zero.

In [ ]:
# Task 3.5 -- Expected Improvement

def expected_improvement(mu, sigma, f_best, xi=0.01):
    # xi prevents EI from collapsing once the model is very confident
    # Phi = stats.norm.cdf, phi = stats.norm.pdf
    # Return 0 wherever sigma == 0
    # Your code here
    pass


## Task 3.6: Implement PI and Compare All Three Acquisition Functions (5 pts)

```
PI(x) = Phi( (mu(x) - f_best - xi) / sigma(x) )
```

Implement `probability_of_improvement` with inline comments.
Plot a 3-panel scatter (one panel per AF) with each candidate colored by score
and the top-10 marked.

**Deliverables:** `acquisition_comparison.png`, written answers to 4 questions.

In [ ]:
# Task 3.6 -- Probability of Improvement and Three-Way Comparison

def probability_of_improvement(mu, sigma, f_best, xi=0.01):
    # Probability that a candidate beats the current best by at least xi
    # Guard the sigma==0 case
    # Your code here
    pass


# Three-panel acquisition comparison plot -> acquisition_comparison.png
# Your code here


### Task 3.6 -- Written Answers

**Q1: Do the three acquisition functions agree on which designs to test next,
or do they identify different regions? What does disagreement tell you?**

*Your answer here*

---

**Q2: Describe a concrete StructureLab scenario where PI's conservatism
would lead to a poor choice that EI would avoid.**

*Your answer here*

---

**Q3: All three AFs reduce to pure exploitation as sigma -> 0 everywhere.
Why does this happen mathematically, and is it the right behavior
when your model is very confident?**

*Your answer here*

---

**Q4: UCB has kappa; EI and PI have xi.
Are they conceptually equivalent tuning parameters?
What is the practical difference in how each affects candidate selection?**

*Your answer here*

## Task 3.8: Run the Campaigns (4 pts)

Run three campaigns, each reset to the same 5 initial samples (`random_state=88`):

1. **UCB** -- kappa=2.0, max 150 iterations, stop when 15 top-5% hits are found
2. **EI** -- xi=0.01, same stopping rule
3. **Random baseline** -- no GP, random candidate each step, run all 150 iterations

Store per-iteration history: cumulative top-5% count, best toughness found, query toughness.

In [ ]:
# Task 3.8 -- BO Campaigns
# IMPORTANT: reset to the same 5 initial samples before each campaign!

# ----- UCB campaign -----
# Your code here

# ----- EI campaign -----
# Your code here

# ----- Random baseline -----
# Your code here


## Task 3.7: Regret Analysis and Theoretical Bounds (5 pts)

Simple regret = f* - best_found_so_far.
Cumulative regret = sum of (f* - toughness_queried) over all steps.
BO theory proves GP-UCB with adaptive kappa_t achieves sublinear cumulative regret.

**[Bonus +3 pts]** Implement adaptive kappa_t = sqrt(2*log(N*t^2*pi^2 / (6*delta))).

**Deliverables:** `bo_simple_regret.png` (with adaptive kappa_t curve),
`bo_cumulative_regret.png`, printed f*.

*Note: run this cell after Task 3.8.*

In [ ]:
# Task 3.7 -- Regret Analysis
# NOTE: Task 3.8 (campaigns) is above this cell -- run it first to populate
# UCB_history, EI_history, and random_history before running this cell.

# f* = global optimum in the full dataset
# Your code here


### Task 3.7 -- Written Answers

**Q1: Engineer A: minimize simple regret.
Engineer B: minimize cumulative regret.
Under what conditions is each engineer right?**

*Your answer here*

---

**Q2: The toughness landscape has varying smoothness across the design space.
What does the theoretical bound imply about campaign performance
in smooth vs. rough regions?**

*Your answer here*

---

**Q3: Does your empirical simple regret curve look roughly like O(1/sqrt(T))?
If BO outperforms the bound, what could explain it?
If it underperforms, what are likely causes?**

*Your answer here*

## Task 3.9: When BO Fails (5 pts)

Before StructureLab scales to their next project, document the failure modes.

1. **Kernel misspecification:** run UCB with RBF kernel; compare simple regret to Matern-UCB.
2. **Non-Gaussian noise:** fit a standard GP to f(x)=sin(x) data with
   Student-t noise (nu=2, scale=0.5); show posterior near outliers.
3. **Surrogate vs. acquisition optimization:** answer conceptual questions.
4. **Campaign design:** batch queries, human override, stopping criterion.

**Deliverables:** `gp_nongaussian_noise.png`, written answers to all items.

In [ ]:
# Task 3.9 -- Failure Modes

# 1. Kernel misspecification: UCB with RBF kernel, record simple regret
# Your code here

# 2. Non-Gaussian noise: Student-t noise data
# f(x) = sin(x), x in [0, 2*pi], noise from stats.t(df=2, scale=0.5)
# Your code here


### Task 3.9 -- Written Answers

**1. Kernel misspecification:** does RBF vs. Matern significantly affect performance?
When would you expect kernel choice to matter much more?

*Your answer here*

---

**2. Non-Gaussian noise:** how does the GP posterior behave near Student-t outliers?
What would you do in practice?

*Your answer here*

---

**3a.** At the surrogate fitting step, what is being maximized and what are the free variables?

*Your answer here*

**3b.** At the acquisition step, what is being maximized and over what space?

*Your answer here*

**3c.** If StructureLab moved to a continuous design space (any real-valued n, theta, r, t),
why would acquisition optimization become hard? What are the standard approaches?

*Your answer here*

---

**4a.** How would you query 5 candidates at once?
What is the simplest approximation and what does it sacrifice?

*Your answer here*

**4b.** An engineer vetoes 3 of your top-10 candidates as geometrically unprintable.
How does this human override interact with the BO framework?

*Your answer here*

**4c.** Describe a stopping criterion that does not require knowing f* in advance.

*Your answer here*

## Task 3.10: The Final Report to StructureLab (4 pts)

**Required plots:**
- `bo_discovery_plot.png` -- all ~1800 designs with top-5% threshold marked
  and found candidates highlighted (UCB and EI in two subplots)
- `bo_cumulative_discovery.png` -- cumulative top-5% found vs. iteration
  (all 3 methods on same axes; dashed line at 15)
- `bo_best_value_curve.png` -- best toughness found vs. iteration (all 3 methods)

Written answers to 5 questions in the answer cell below.

In [ ]:
# Task 3.10 -- Final Report Plots
# bo_discovery_plot.png, bo_cumulative_discovery.png, bo_best_value_curve.png
# Your code here


### Task 3.10 -- Written Answers

**Q1: How many iterations did each method need to find 15 top-5% candidates?
Did any fail within 150 iterations?
How does this compare to the random baseline expected number from Task 3.1?**

*Your answer here*

---

**Q2: Which acquisition function found the most top-5% designs within 150 tests?
Was this surprising given the theoretical properties of UCB vs. EI?**

*Your answer here*

---

**Q3: Did one method find candidates early and plateau while the other found them gradually?
What does this tell you about their exploration-exploitation balance?**

*Your answer here*

---

**Q4: Does BO meaningfully outperform random search on this dataset?
If the gap is smaller than expected, what properties of the crossed-barrel landscape
could explain it?**

*Your answer here*

---

**Q5: Write a two-paragraph recommendation for StructureLab's next campaign
(different lattice topology, same budget).
Address: which acquisition function, how to set kappa or xi,
how many initial random tests, and when to stop.**

*Your answer here*

---
# Part 4: Bayesian Hyperparameter Tuning with Optuna (15 pts)

**Story:** A national lab ceramics group is developing high-entropy oxide (HEO) ceramics
for concentrated solar power (CSP) thermal energy storage.
The key figure of merit is heat capacity Cp.
You need a fast predictive model to screen 500+ candidates
before committing to 3-week synthesis runs.
Training data: `data/cp_data_cleaned.csv`.

Optuna's TPE sampler is itself a form of Bayesian optimization.
By the end of this part you will be able to map it directly
onto the GP-UCB framework from Part 3.

See **HW4_instructions.md -- Part 4** for the complete scenario and requirements.

## Task 4.1: Understand the Dataset (Tasks 4.1-4.2 graded together: 3 pts)

Load `data/cp_data_cleaned.csv`. Print shape, column names,
summary statistics for Cp and T,
number of unique formulas, and average T measurements per formula.

In [ ]:
# Task 4.1 -- Explore the HEO Cp Dataset
df_cp = pd.read_csv('data/cp_data_cleaned.csv')
# Your code here


### Task 4.1 -- Written Answer

**If you used a naive random 80/20 split, could test rows leak information
from the same compound's training rows?
Why is this a problem for evaluating whether the model generalizes
to new compounds?**

*Your answer here*

## Task 4.2: Build Features and a Leakage-Free Train/Test Split

Generate CBFV features with `elem_prop='oliynyk'` and `drop_duplicates=False`.
Check the `skipped` return value and realign T to `X_comp.index` before appending.
Split by formula using `GroupShuffleSplit(test_size=0.2, random_state=88)`.

In [ ]:
# Task 4.2 -- CBFV Features + Group-Based Split
from cbfv.composition import generate_features

# generate_features returns (X_comp, y_comp, formulae, skipped)
# Drop skipped rows and realign T column to X_comp.index before appending
# GroupShuffleSplit keeps all rows of the same formula in the same split
# Your code here


### Task 4.2 -- Written Answer

**What would the test MAE look like if you used a random split instead of a group split?
Would it be higher or lower than the group-split MAE, and why?**

*Your answer here*

## Task 4.3: Define the Optuna Objective Function (4 pts)

Write `objective(trial)` with an inline comment on each hyperparameter line
explaining why that Optuna sampling method is appropriate
(e.g., why `suggest_int` vs. `suggest_categorical`).

Parameters: `n_estimators` (int 50-500), `max_depth` (categorical: None, 5, 10, 20, 50),
`min_samples_split` (int 2-20), `min_samples_leaf` (int 1-10),
`max_features` (categorical: sqrt, log2, 0.5, 1.0).
Return test-set MAE.

In [ ]:
# Task 4.3 -- Optuna Objective Function

def objective(trial):
    # Optuna calls this function once per trial.
    # The TPE sampler uses past results to decide what to try next --
    # exactly like the GP-acquisition loop in Part 3, but in HP space.

    n_estimators = trial.suggest_int('n_estimators', 50, 500)  # integer -- use suggest_int
    # Add remaining hyperparameters with inline comments
    # Your code here
    pass


## Task 4.4: Run the Study (2 pts)

Suppress logging, create a minimization study, and run 50 trials.
Print the best trial number, best MAE, and best hyperparameters.

In [ ]:
# Task 4.4 -- Run the Optuna Study
optuna.logging.set_verbosity(optuna.logging.WARNING)
# Your code here


### Task 4.4 -- Written Answer

**After 50 trials, how many total hyperparameter combinations were evaluated?
If you had used a 5-value grid search over all 5 parameters instead,
how many combinations would that require?**

*Your answer here*

## Task 4.5: Evaluate the Best Model (3 pts)

Retrain a fresh `RandomForestRegressor` with `study.best_params`
and `random_state=88` on the full training set.
Report MAE and R2.
Save `optuna_rf_parity_plot.png` and `optuna_optimization_history.png`.

In [ ]:
# Task 4.5 -- Evaluate Best Model
# Retrain from scratch on full training set (not just one trial's split)
# Your code here


### Task 4.5 -- Written Answer

**Does the optimization history show a clear improvement trend, or does it plateau quickly?
What does this tell you about the difficulty of this hyperparameter landscape?**

*Your answer here*

## Task 4.6: Situate the Result (3 pts)

Help the HEO solar group decide: RF, CrabNet, or something else?
This task is written answers only -- no code required.

### Task 4.6 -- Written Answers

**Q1: Report your RF test-set MAE and R2.
Recall CrabNet Cp results from HW3 Task 3.7.
HW3 CrabNet used only T=298K (~243 samples);
this RF uses all temperatures (~4500 rows).
Does that difference make the comparison fair?
If not, how would you redesign the comparison?**

*Your answer here*

---

**Q2: Compare RF and CrabNet across five dimensions:
model complexity, interpretability, data requirements,
physical priors encoded, and ease of adding new compounds.**

*Your answer here*

---

**Q3: The solar group wants to screen 500 new HEO compositions across 20 temperatures each
(10,000 queries), with 2 weeks and no GPU.
Which model do you recommend, and why?**

*Your answer here*

---

**Q4: Optuna's TPE sampler is itself a form of Bayesian optimization.
Using the vocabulary from Part 3:
what plays the role of the surrogate model in TPE?
What plays the role of the acquisition function?
How does TPE differ structurally from the GP-UCB loop --
specifically, what does it use instead of a GP,
and what does it maximize instead of UCB?**

*Your answer here*